# Phase 5: Activation Patching — Causal Localization of Emotions

**Causal intervention to determine WHERE each emotion's information lives inside BERT.**

## Methodology

Traditional lesion studies (Notebook 08) show **what breaks** when we compress a layer — but they cannot distinguish between two hypotheses:
1. The layer **computes** critical information (it is the source)
2. The layer merely **transmits** information computed elsewhere (it is on the path)

**Activation patching** (Meng et al., 2022; *Locating and Editing Factual Associations in GPT*) resolves this ambiguity through causal intervention:

1. Create an aggressively compressed model (rank 64) that has degraded performance
2. For each encoder layer L (0-11):
   - Run the **baseline** model and cache the hidden state output of layer L
   - Run the **compressed** model, but **replace** layer L's output with the baseline's cached activation
   - Measure: does this single-layer restoration recover per-emotion F1?
3. The layer whose patching restores the most F1 is where that emotion's critical information was **destroyed by compression**

This is **causal evidence**, not correlation. If patching layer L restores emotion E, it means:
- Layer L's computation is necessary for emotion E
- The compressed version of layer L has lost the specific information needed for E
- No other layer can compensate for this loss

We also perform **component-level patching** (attention vs FFN) to further localize within each layer.

In [ ]:
import sys, os, warnings, time, gc
warnings.filterwarnings('ignore')

# -- Project root (Colab or local) --
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    get_target_layer_names,
    filter_layer_names,
)

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']

# ================================================================
#  DARK THEME + NEON PALETTE (consistent with notebook 08)
# ================================================================

DARK_BG        = '#0d1117'
DARK_GRID      = '#21262d'
TEXT_PRIMARY   = '#e6edf3'
TEXT_SECONDARY = '#8b949e'

NEON_CMAP = LinearSegmentedColormap.from_list(
    'neon', ['#00ff87', '#00d4ff', '#7b2ff7', '#ff006e'], N=256)

RESTORATION_CMAP = LinearSegmentedColormap.from_list(
    'restoration', ['#0d1117', '#1a1a2e', '#16213e', '#0f3460',
                    '#00d4ff', '#00ff87', '#ffd60a', '#ff006e'], N=256)

plt.rcParams.update({
    'figure.facecolor':  DARK_BG,
    'axes.facecolor':    DARK_BG,
    'text.color':        TEXT_PRIMARY,
    'axes.labelcolor':   TEXT_PRIMARY,
    'xtick.color':       TEXT_SECONDARY,
    'ytick.color':       TEXT_SECONDARY,
    'axes.edgecolor':    DARK_GRID,
    'grid.color':        DARK_GRID,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.facecolor': DARK_BG,
    'savefig.bbox':      'tight',
    'font.size':         12,
})

results_dir = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(results_dir, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')
print('Dark theme configured')

In [ ]:
# ================================================================
#  LOAD BASELINE MODEL, COMPRESSED MODEL, AND TEST DATA
# ================================================================

model_path = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-23emo-final')

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_path, problem_type='multi_label_classification',
)
baseline_model.eval()
baseline_model.to(device)
print(f'Baseline model loaded from {model_path}')

# Aggressive compression: rank 64
PATCH_RANK = 64
all_target_names = get_target_layer_names(baseline_model)
compressed_model = apply_svd_compression(baseline_model, rank=PATCH_RANK, layer_names=all_target_names)
compressed_model.eval()
compressed_model.to(device)
print(f'Compressed model created (rank={PATCH_RANK}, all layers)')

# Load test data
_, _, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS
)
num_labels = len(emotion_names)  # 23
print(f'Test set: {len(test_ds)} examples')

# Evaluate both models to get baseline and compressed per-emotion F1
eval_args = TrainingArguments(
    output_dir=os.path.join(results_dir, 'tmp_eval'),
    per_device_eval_batch_size=64,
    fp16=(device == 'cuda'),
    report_to='none',
)

print('\nEvaluating baseline...')
baseline_trainer = Trainer(
    model=baseline_model, args=eval_args,
    compute_metrics=compute_metrics, data_collator=data_collator,
)
baseline_metrics = baseline_trainer.evaluate(test_ds)
baseline_per_emotion = np.array([baseline_metrics[f'eval_f1_label_{i}'] for i in range(num_labels)])
baseline_f1_macro = baseline_metrics['eval_f1_macro']
del baseline_trainer

print('Evaluating compressed...')
compressed_trainer = Trainer(
    model=compressed_model, args=eval_args,
    compute_metrics=compute_metrics, data_collator=data_collator,
)
compressed_metrics = compressed_trainer.evaluate(test_ds)
compressed_per_emotion = np.array([compressed_metrics[f'eval_f1_label_{i}'] for i in range(num_labels)])
compressed_f1_macro = compressed_metrics['eval_f1_macro']
del compressed_trainer

print(f'\nBaseline  F1 macro: {baseline_f1_macro:.4f}')
print(f'Compressed F1 macro: {compressed_f1_macro:.4f}')
print(f'Degradation: {baseline_f1_macro - compressed_f1_macro:.4f}')
print(f'\nPer-emotion F1 range:')
print(f'  Baseline:   [{baseline_per_emotion.min():.4f}, {baseline_per_emotion.max():.4f}]')
print(f'  Compressed: [{compressed_per_emotion.min():.4f}, {compressed_per_emotion.max():.4f}]')

## 1. Layer-Level Activation Patching

In [ ]:
@torch.no_grad()
def run_patched_inference(baseline_model, compressed_model, dataset, data_collator,
                          patch_layer_idx, patch_component='all', batch_size=64):
    """Run compressed model with activations patched from baseline at one layer.

    patch_component: 'all' (entire layer output), 'attention', or 'ffn'

    Returns per-emotion F1 array (28,) and F1 macro scalar.
    """
    dev = next(baseline_model.parameters()).device
    all_logits = []
    all_labels = []

    for i in range(0, len(dataset), batch_size):
        batch_indices = range(i, min(i + batch_size, len(dataset)))
        batch = data_collator([dataset[j] for j in batch_indices])
        input_ids = batch['input_ids'].to(dev)
        attention_mask = batch['attention_mask'].to(dev)

        # Step 1: Get baseline activations at the target layer/component
        baseline_cache = {}

        if patch_component == 'all':
            # Patch entire BertLayer output (returns tuple)
            def save_hook(module, input, output):
                baseline_cache['output'] = output
            handle = baseline_model.bert.encoder.layer[patch_layer_idx].register_forward_hook(save_hook)
            baseline_model(input_ids=input_ids, attention_mask=attention_mask)
            handle.remove()

            # Step 2: Run compressed with patch
            def patch_hook(module, input, output):
                return baseline_cache['output']
            handle = compressed_model.bert.encoder.layer[patch_layer_idx].register_forward_hook(patch_hook)
            outputs = compressed_model(input_ids=input_ids, attention_mask=attention_mask)
            handle.remove()

        elif patch_component == 'attention':
            # Patch only attention block output (BertAttention returns tuple)
            def save_hook(module, input, output):
                baseline_cache['output'] = output
            handle = baseline_model.bert.encoder.layer[patch_layer_idx].attention.register_forward_hook(save_hook)
            baseline_model(input_ids=input_ids, attention_mask=attention_mask)
            handle.remove()

            def patch_hook(module, input, output):
                return baseline_cache['output']
            handle = compressed_model.bert.encoder.layer[patch_layer_idx].attention.register_forward_hook(patch_hook)
            outputs = compressed_model(input_ids=input_ids, attention_mask=attention_mask)
            handle.remove()

        elif patch_component == 'ffn':
            # Patch FFN output (BertOutput: dense + LayerNorm + residual)
            def save_hook(module, input, output):
                baseline_cache['output'] = output
            handle = baseline_model.bert.encoder.layer[patch_layer_idx].output.register_forward_hook(save_hook)
            baseline_model(input_ids=input_ids, attention_mask=attention_mask)
            handle.remove()

            def patch_hook(module, input, output):
                return baseline_cache['output']
            handle = compressed_model.bert.encoder.layer[patch_layer_idx].output.register_forward_hook(patch_hook)
            outputs = compressed_model(input_ids=input_ids, attention_mask=attention_mask)
            handle.remove()

        all_logits.append(outputs.logits.cpu())
        all_labels.append(batch['labels'])

    # Compute per-emotion F1
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    per_emotion_f1 = f1_score(labels, preds, average=None, zero_division=0)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    return per_emotion_f1, f1_macro


print('Patching function defined.')

In [ ]:
# ================================================================
#  RUN LAYER-LEVEL PATCHING FOR ALL 12 LAYERS
# ================================================================

layer_patched_f1 = np.zeros((12, num_labels))    # per-emotion F1 when patching layer L
layer_patched_macro = np.zeros(12)                # macro F1 when patching layer L

print(f'Layer-level activation patching (rank={PATCH_RANK})')
print('=' * 60)
t_total = time.time()

for layer_idx in range(12):
    t0 = time.time()
    per_emo, f1_m = run_patched_inference(
        baseline_model, compressed_model, test_ds, data_collator,
        patch_layer_idx=layer_idx, patch_component='all',
    )
    layer_patched_f1[layer_idx] = per_emo
    layer_patched_macro[layer_idx] = f1_m
    elapsed = time.time() - t0
    restoration = (f1_m - compressed_f1_macro) / max(baseline_f1_macro - compressed_f1_macro, 1e-8)
    print(f'  Layer {layer_idx:2d}: F1 = {f1_m:.4f}  '
          f'(restoration = {restoration:.1%})  [{elapsed:.1f}s]')

print(f'\nCompleted in {time.time() - t_total:.0f}s')
best_layer = np.argmax(layer_patched_macro)
print(f'Best single-layer patch: Layer {best_layer} '
      f'(F1 = {layer_patched_macro[best_layer]:.4f})')

## 2. Restoration Map

In [ ]:
# ================================================================
#  COMPUTE RESTORATION SCORES
#  restoration = (patched_f1 - compressed_f1) / (baseline_f1 - compressed_f1)
#  Clamped to [0, 1]. Measures fraction of lost performance restored.
# ================================================================

f1_gap = baseline_per_emotion - compressed_per_emotion  # (23,)
# Avoid division by zero for emotions where compression caused no damage
safe_gap = np.where(np.abs(f1_gap) > 0.005, f1_gap, 1.0)

# restoration_matrix: (12 layers, 23 emotions)
restoration_matrix = (layer_patched_f1 - compressed_per_emotion[np.newaxis, :]) / safe_gap[np.newaxis, :]
restoration_matrix = np.clip(restoration_matrix, 0.0, 1.0)

# For emotions with no gap, set restoration to 0 (nothing to restore)
no_gap_mask = np.abs(f1_gap) <= 0.005
restoration_matrix[:, no_gap_mask] = 0.0

# Filter out emotions with near-zero baseline F1
valid_mask = baseline_per_emotion > 0.01
valid_indices = np.where(valid_mask)[0]
valid_emotions = [emotion_names[i] for i in valid_indices]
n_valid = len(valid_emotions)

restoration_valid = restoration_matrix[:, valid_indices]  # (12, n_valid)

# Sort emotions by their "critical layer" (layer with highest restoration)
critical_layer_per_emotion = np.argmax(restoration_valid, axis=0)  # (n_valid,)
sort_idx = np.argsort(critical_layer_per_emotion)
sorted_emotions = [valid_emotions[i] for i in sort_idx]
restoration_sorted = restoration_valid[:, sort_idx]  # (12, n_valid_sorted)

print(f'Valid emotions: {n_valid} (excluded {num_labels - n_valid} with baseline F1 ~ 0)')
print(f'\nCritical layers per emotion:')
for i, emo in enumerate(sorted_emotions):
    cl = critical_layer_per_emotion[sort_idx[i]]
    rs = restoration_sorted[cl, i]
    print(f'  {emo:18s}: Layer {cl:2d}  (restoration = {rs:.2f})')

In [ ]:
# ================================================================
#  HEATMAP: Causal Restoration Map
#  28 emotions x 12 layers showing restoration score
# ================================================================

fig, ax = plt.subplots(figsize=(20, 10))

im = ax.imshow(restoration_sorted, cmap=RESTORATION_CMAP, vmin=0, vmax=1,
               aspect='auto', interpolation='nearest')

ax.set_yticks(range(12))
ax.set_yticklabels([f'Layer {i}' for i in range(12)], fontsize=12, fontweight='bold')
ax.set_xticks(range(len(sorted_emotions)))
ax.set_xticklabels(sorted_emotions, rotation=55, ha='right', fontsize=10)

# Annotate percentages
for i in range(12):
    for j in range(len(sorted_emotions)):
        val = restoration_sorted[i, j]
        if val > 0.02:
            txt_color = '#000000' if val > 0.6 else TEXT_PRIMARY
            ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                    fontsize=6, color=txt_color, fontweight='bold')

# Colorbar
cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Restoration Score', fontsize=13, color=TEXT_PRIMARY, labelpad=12)
cbar.ax.tick_params(colors=TEXT_SECONDARY, labelsize=11)
cbar.outline.set_edgecolor(DARK_GRID)

ax.set_title('CAUSAL RESTORATION MAP\nWhere Each Emotion\'s Information Lives',
             fontsize=22, fontweight='bold', pad=18,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
fig.text(0.45, 0.02,
         f'Activation patching: baseline activations injected into compressed model (rank {PATCH_RANK})  |  '
         f'Sorted by critical layer  |  {n_valid} emotions',
         fontsize=10, color=TEXT_SECONDARY, ha='center',
         path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

plt.savefig(os.path.join(results_dir, 'activation_patching_01_restoration_map.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print('Saved: results/activation_patching_01_restoration_map.png')

In [ ]:
# ================================================================
#  BAR CHART: Mean restoration per layer
# ================================================================

mean_restoration_per_layer = restoration_valid.mean(axis=1)  # (12,)

fig, ax = plt.subplots(figsize=(14, 6))

colors_bar = [NEON_CMAP(val / mean_restoration_per_layer.max())
              for val in mean_restoration_per_layer]

bars = ax.bar(range(12), mean_restoration_per_layer, color=colors_bar,
              edgecolor=DARK_GRID, linewidth=1.5, width=0.75)

# Annotate bars
for idx, (bar, val) in enumerate(zip(bars, mean_restoration_per_layer)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11,
            color=TEXT_PRIMARY, fontweight='bold')

ax.set_xlabel('Encoder Layer', fontsize=14, labelpad=10)
ax.set_ylabel('Mean Restoration Score', fontsize=14, labelpad=10)
ax.set_xticks(range(12))
ax.set_xticklabels([f'L{i}' for i in range(12)], fontsize=13, fontweight='bold')
ax.set_ylim(0, mean_restoration_per_layer.max() * 1.15)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='y', alpha=0.08, color='#ffffff')

ax.set_title('MEAN CAUSAL RESTORATION PER LAYER\n'
             'Which layers carry the most emotion-critical information?',
             fontsize=18, fontweight='bold', pad=15,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

plt.savefig(os.path.join(results_dir, 'activation_patching_02_mean_restoration.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print('Saved: results/activation_patching_02_mean_restoration.png')

## 3. Component-Level Patching

In [ ]:
# ================================================================
#  COMPONENT-LEVEL PATCHING
#  For the top-4 most important layers, patch attention-only vs FFN-only
# ================================================================

# Select top-4 layers by mean restoration
top_k = 4
top_layers = np.argsort(mean_restoration_per_layer)[-top_k:][::-1]
print(f'Top {top_k} layers for component-level patching: {list(top_layers)}')
print(f'Mean restoration: {[f"{mean_restoration_per_layer[l]:.3f}" for l in top_layers]}')

# Storage: {(layer_idx, component): (per_emotion_f1, f1_macro)}
component_patched = {}

print(f'\nComponent-level activation patching')
print('=' * 60)
t_total = time.time()

for layer_idx in top_layers:
    for component in ['attention', 'ffn']:
        t0 = time.time()
        per_emo, f1_m = run_patched_inference(
            baseline_model, compressed_model, test_ds, data_collator,
            patch_layer_idx=layer_idx, patch_component=component,
        )
        component_patched[(layer_idx, component)] = (per_emo, f1_m)
        elapsed = time.time() - t0
        restoration = (f1_m - compressed_f1_macro) / max(baseline_f1_macro - compressed_f1_macro, 1e-8)
        print(f'  Layer {layer_idx:2d} [{component:9s}]: F1 = {f1_m:.4f}  '
              f'(restoration = {restoration:.1%})  [{elapsed:.1f}s]')

print(f'\nCompleted in {time.time() - t_total:.0f}s')

# Clean up
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# ================================================================
#  GROUPED BAR CHART: Attention vs FFN vs Full Layer Patch
# ================================================================

fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(top_layers))
width = 0.25

macro_gap = max(baseline_f1_macro - compressed_f1_macro, 1e-8)

# Compute restoration for each configuration
full_rest = [(layer_patched_macro[l] - compressed_f1_macro) / macro_gap for l in top_layers]
attn_rest = [(component_patched[(l, 'attention')][1] - compressed_f1_macro) / macro_gap
             for l in top_layers]
ffn_rest = [(component_patched[(l, 'ffn')][1] - compressed_f1_macro) / macro_gap
            for l in top_layers]

bars1 = ax.bar(x - width, full_rest, width, label='Full Layer',
               color='#00ff87', edgecolor=DARK_GRID, linewidth=1.2)
bars2 = ax.bar(x, attn_rest, width, label='Attention Only',
               color='#00d4ff', edgecolor=DARK_GRID, linewidth=1.2)
bars3 = ax.bar(x + width, ffn_rest, width, label='FFN Only',
               color='#ff006e', edgecolor=DARK_GRID, linewidth=1.2)

# Annotate
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.005,
                f'{height:.1%}', ha='center', va='bottom',
                fontsize=9, color=TEXT_PRIMARY, fontweight='bold')

ax.set_xlabel('Layer', fontsize=14, labelpad=10)
ax.set_ylabel('Restoration Score (F1 Macro)', fontsize=14, labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels([f'Layer {l}' for l in top_layers], fontsize=13, fontweight='bold')

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='y', alpha=0.08, color='#ffffff')

ax.legend(fontsize=13, loc='upper right', framealpha=0.15,
          edgecolor=DARK_GRID, labelcolor=TEXT_PRIMARY, fancybox=True)

ax.set_title('COMPONENT-LEVEL CAUSAL RESTORATION\n'
             'Attention vs FFN: where does the information live within each layer?',
             fontsize=18, fontweight='bold', pad=15,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

plt.savefig(os.path.join(results_dir, 'activation_patching_03_component_bars.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print('Saved: results/activation_patching_03_component_bars.png')

## 4. Per-Emotion Causal Profiles

In [ ]:
# ================================================================
#  PER-EMOTION RESTORATION CURVES
#  Select 8 emotions: mix of fragile, robust, and interesting patterns
# ================================================================

# Identify fragile (large gap), robust (small gap), and interesting emotions
emotion_gap = baseline_per_emotion - compressed_per_emotion  # (23,)
valid_gaps = [(i, emotion_gap[i]) for i in valid_indices]
valid_gaps.sort(key=lambda x: -x[1])  # largest gap first

# Pick 4 most fragile (largest degradation from compression)
fragile = [emotion_names[idx] for idx, _ in valid_gaps[:4]]
# Pick 4 most robust among those with some gap
robust_candidates = [(idx, gap) for idx, gap in valid_gaps if gap > 0.005]
robust = [emotion_names[idx] for idx, _ in robust_candidates[-4:]]

selected_emotions = fragile + robust
# Deduplicate while preserving order
seen = set()
selected_emotions = [e for e in selected_emotions if not (e in seen or seen.add(e))]
selected_emotions = selected_emotions[:8]

print(f'Selected emotions for causal profiles:')
for emo in selected_emotions:
    idx = emotion_names.index(emo)
    print(f'  {emo:18s}: baseline={baseline_per_emotion[idx]:.4f}  '
          f'compressed={compressed_per_emotion[idx]:.4f}  '
          f'gap={emotion_gap[idx]:.4f}')

# Plot restoration curves
fig, ax = plt.subplots(figsize=(16, 8))

palette = ['#ff006e', '#ff4d6d', '#ff9500', '#ffd60a',
           '#00ff87', '#00d4ff', '#7b2ff7', '#b388ff']

for i, emo in enumerate(selected_emotions):
    emo_idx = emotion_names.index(emo)
    # Per-layer restoration for this emotion
    gap = baseline_per_emotion[emo_idx] - compressed_per_emotion[emo_idx]
    if abs(gap) > 0.005:
        rest_curve = (layer_patched_f1[:, emo_idx] - compressed_per_emotion[emo_idx]) / gap
        rest_curve = np.clip(rest_curve, 0.0, 1.0)
    else:
        rest_curve = np.zeros(12)

    color = palette[i % len(palette)]
    ax.plot(range(12), rest_curve, '-o', color=color, linewidth=2.5,
            markersize=7, label=emo, alpha=0.9,
            path_effects=[pe.withStroke(linewidth=4, foreground=color + '30')])

    # Annotate critical layer (peak)
    peak_layer = np.argmax(rest_curve)
    peak_val = rest_curve[peak_layer]
    if peak_val > 0.05:
        ax.annotate(f'L{peak_layer}', xy=(peak_layer, peak_val),
                    xytext=(5, 8), textcoords='offset points',
                    fontsize=9, color=color, fontweight='bold',
                    path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

ax.set_xlabel('Encoder Layer', fontsize=14, labelpad=10)
ax.set_ylabel('Restoration Score', fontsize=14, labelpad=10)
ax.set_xticks(range(12))
ax.set_xticklabels([f'L{i}' for i in range(12)], fontsize=13, fontweight='bold')
ax.set_ylim(-0.05, 1.1)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.grid(True, axis='y', alpha=0.08, color='#ffffff')

ax.legend(fontsize=11, loc='upper left', framealpha=0.15,
          edgecolor=DARK_GRID, labelcolor=TEXT_PRIMARY, fancybox=True,
          ncol=2)

ax.set_title('PER-EMOTION CAUSAL PROFILES\n'
             'Restoration curves reveal where each emotion\'s information concentrates',
             fontsize=18, fontweight='bold', pad=15,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

plt.savefig(os.path.join(results_dir, 'activation_patching_04_per_emotion_curves.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print('Saved: results/activation_patching_04_per_emotion_curves.png')

In [ ]:
# ================================================================
#  CAUSAL BRAIN SCAN: 2x2 grid for 4 representative emotions
#  Layer x Component (attention / FFN) restoration heatmap
# ================================================================

# Select 4 emotions with distinct patterns
max_restoration_per_emotion = restoration_valid.max(axis=0)  # (n_valid,)
mean_restoration_per_emotion = restoration_valid.mean(axis=0)

# 1. Most localized (highest peak restoration)
idx_localized = int(np.argmax(max_restoration_per_emotion))
# 2. Most distributed (highest mean, but low peak-to-mean ratio)
spread = mean_restoration_per_emotion.copy()
spread[idx_localized] = -1
idx_distributed = int(np.argmax(spread))
# 3. Most fragile (largest f1 gap among valid)
gaps_valid = [emotion_gap[valid_indices[i]] for i in range(n_valid)]
idx_fragile = int(np.argmax(gaps_valid))
if idx_fragile in [idx_localized, idx_distributed]:
    gaps_copy = list(gaps_valid)
    gaps_copy[idx_localized] = -1
    gaps_copy[idx_distributed] = -1
    idx_fragile = int(np.argmax(gaps_copy))
# 4. Most robust (smallest gap but still some restoration)
robust_scores = mean_restoration_per_emotion.copy()
for idx in [idx_localized, idx_distributed, idx_fragile]:
    robust_scores[idx] = np.inf
# Pick the one with lowest mean restoration that still has some signal
candidates_robust = [(i, robust_scores[i]) for i in range(n_valid)
                     if robust_scores[i] < np.inf and max_restoration_per_emotion[i] > 0.01]
if candidates_robust:
    idx_robust = min(candidates_robust, key=lambda x: x[1])[0]
else:
    idx_robust = 0

brain_scan_indices = [idx_localized, idx_fragile, idx_distributed, idx_robust]
brain_scan_labels = ['LOCALIZED', 'FRAGILE', 'DISTRIBUTED', 'ROBUST']
brain_scan_colors = ['#ff006e', '#ff9500', '#00d4ff', '#00ff87']

print('Selected emotions for brain scan:')
for i, (vidx, label, color) in enumerate(zip(brain_scan_indices, brain_scan_labels, brain_scan_colors)):
    emo = valid_emotions[vidx]
    print(f'  {label:12s}: {emo} (peak restoration = {max_restoration_per_emotion[vidx]:.3f})')

# Build 2x2 figure
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, vidx, label, accent in zip(axes.flat, brain_scan_indices, brain_scan_labels, brain_scan_colors):
    emo_name = valid_emotions[vidx]
    emo_full_idx = valid_indices[vidx]

    # Build grid: 12 layers x 3 components (full, attention, ffn)
    # For layers in top_layers, we have component data; for others, use full-layer only
    grid = np.zeros((12, 3))  # columns: Full, Attention, FFN

    gap = baseline_per_emotion[emo_full_idx] - compressed_per_emotion[emo_full_idx]
    safe_g = gap if abs(gap) > 0.005 else 1.0

    for l in range(12):
        full_r = (layer_patched_f1[l, emo_full_idx] - compressed_per_emotion[emo_full_idx]) / safe_g
        grid[l, 0] = np.clip(full_r, 0, 1)

        if (l, 'attention') in component_patched:
            attn_r = (component_patched[(l, 'attention')][0][emo_full_idx] - compressed_per_emotion[emo_full_idx]) / safe_g
            grid[l, 1] = np.clip(attn_r, 0, 1)
        if (l, 'ffn') in component_patched:
            ffn_r = (component_patched[(l, 'ffn')][0][emo_full_idx] - compressed_per_emotion[emo_full_idx]) / safe_g
            grid[l, 2] = np.clip(ffn_r, 0, 1)

    im = ax.imshow(grid, cmap=RESTORATION_CMAP, vmin=0, vmax=1,
                   aspect='auto', interpolation='nearest')

    ax.set_yticks(range(12))
    ax.set_yticklabels([f'L{i}' for i in range(12)], fontsize=9)
    ax.set_xticks(range(3))
    ax.set_xticklabels(['Full Layer', 'Attention', 'FFN'],
                       fontsize=11, fontweight='bold')
    ax.get_xticklabels()[0].set_color('#00ff87')
    ax.get_xticklabels()[1].set_color('#00d4ff')
    ax.get_xticklabels()[2].set_color('#ff006e')

    # Annotate values
    for row in range(12):
        for col in range(3):
            val = grid[row, col]
            if val > 0.02:
                txt_color = '#000000' if val > 0.6 else TEXT_PRIMARY
                ax.text(col, row, f'{val:.0%}', ha='center', va='center',
                        fontsize=8, color=txt_color, fontweight='bold')

    # Grid lines
    ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 12, 1), minor=True)
    ax.grid(which='minor', color=DARK_BG, linewidth=1.5)
    ax.tick_params(which='minor', size=0)

    # Mark critical layer
    peak_row = np.argmax(grid[:, 0])
    ax.plot(-0.3, peak_row, '>', markersize=10, color=accent, clip_on=False)

    ax.set_title(f'{emo_name.upper()}\n{label}',
                 fontsize=14, fontweight='bold', pad=12, color=accent,
                 path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

fig.suptitle('CAUSAL BRAIN SCAN\nActivation patching reveals each emotion\'s neural footprint',
             fontsize=20, fontweight='bold', y=1.01,
             path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])
fig.text(0.5, 0.975,
         'Brighter = higher restoration  |  Arrow = critical layer  |  '
         'Attention/FFN shown for top layers only',
         fontsize=10, color=TEXT_SECONDARY, ha='center',
         path_effects=[pe.withStroke(linewidth=2, foreground=DARK_BG)])

# Shared colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Restoration Score', fontsize=11, color=TEXT_PRIMARY)
cbar.ax.tick_params(colors=TEXT_SECONDARY)
cbar.outline.set_edgecolor(DARK_GRID)

plt.tight_layout(rect=[0, 0, 0.91, 0.96])
plt.savefig(os.path.join(results_dir, 'activation_patching_05_brain_scan.png'),
            dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
plt.show()
print('Saved: results/activation_patching_05_brain_scan.png')

## 5. Validation: Patching vs Lesion vs Probing

In [ ]:
# ================================================================
#  CROSS-VALIDATION: Patching restoration vs Lesion sensitivity
#  If lesion results exist, correlate with patching results
# ================================================================

lesion_csv = os.path.join(results_dir, 'lesion_per_layer.csv')
has_lesion = os.path.exists(lesion_csv)

if has_lesion:
    df_lesion = pd.read_csv(lesion_csv)
    print(f'Loaded lesion results: {lesion_csv} ({len(df_lesion)} rows)')

    # Build lesion sensitivity matrix: 1 - retention (how much damage)
    lesion_sensitivity = np.zeros((12, num_labels))
    for _, row in df_lesion.iterrows():
        l = int(row['layer'])
        if row['emotion'] in emotion_names:
            emo_idx = emotion_names.index(row['emotion'])
            if pd.notna(row['retention']):
                lesion_sensitivity[l, emo_idx] = 1.0 - row['retention']

    # Compare with patching restoration (both are 12 x 23 matrices)
    # Use only valid emotions
    lesion_valid = lesion_sensitivity[:, valid_indices].flatten()
    patching_valid = restoration_valid.flatten()

    # Scatter plot
    fig, ax = plt.subplots(figsize=(10, 10))

    ax.scatter(lesion_valid, patching_valid, s=15, alpha=0.5,
               color='#00d4ff', edgecolor='none')

    # Compute correlation
    from scipy.stats import pearsonr, spearmanr
    r_pearson, p_pearson = pearsonr(lesion_valid, patching_valid)
    r_spearman, p_spearman = spearmanr(lesion_valid, patching_valid)

    # Fit line
    mask = (lesion_valid > 0) | (patching_valid > 0)
    if mask.sum() > 2:
        z = np.polyfit(lesion_valid[mask], patching_valid[mask], 1)
        p = np.poly1d(z)
        x_line = np.linspace(0, lesion_valid.max(), 100)
        ax.plot(x_line, p(x_line), '--', color='#ff006e', linewidth=2, alpha=0.7)

    ax.set_xlabel('Lesion Sensitivity (1 - retention)', fontsize=14, labelpad=10)
    ax.set_ylabel('Patching Restoration Score', fontsize=14, labelpad=10)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.grid(True, alpha=0.08, color='#ffffff')

    ax.text(0.05, 0.95,
            f'Pearson r = {r_pearson:.3f} (p = {p_pearson:.2e})\n'
            f'Spearman r = {r_spearman:.3f} (p = {p_spearman:.2e})',
            transform=ax.transAxes, fontsize=12, color=TEXT_PRIMARY,
            verticalalignment='top', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor=DARK_GRID, alpha=0.8))

    ax.set_title('PATCHING vs LESION VALIDATION\n'
                 'Do layers that cause most damage when compressed also restore most when patched?',
                 fontsize=16, fontweight='bold', pad=15,
                 path_effects=[pe.withStroke(linewidth=3, foreground=DARK_BG)])

    plt.savefig(os.path.join(results_dir, 'activation_patching_06_validation.png'),
                dpi=300, facecolor=DARK_BG, bbox_inches='tight', pad_inches=0.5)
    plt.show()
    print('Saved: results/activation_patching_06_validation.png')
    print(f'\nCorrelation: Pearson r={r_pearson:.3f}, Spearman r={r_spearman:.3f}')
    print('Positive correlation confirms that patching and lesion studies identify the same critical layers.')
else:
    print(f'Lesion results not found at {lesion_csv}')
    print('Run notebook 08 (emotional map) first to generate lesion data for cross-validation.')
    print('Skipping validation plot.')

## 6. Save Results

In [ ]:
# ================================================================
#  SAVE ALL RESULTS AS CSVs
# ================================================================

# 1. Layer-level patching results (per emotion)
rows_layer = []
for layer_idx in range(12):
    for i, emo in enumerate(emotion_names):
        gap = baseline_per_emotion[i] - compressed_per_emotion[i]
        safe_g = gap if abs(gap) > 0.005 else 1.0
        rest = np.clip((layer_patched_f1[layer_idx, i] - compressed_per_emotion[i]) / safe_g, 0, 1)
        if abs(gap) <= 0.005:
            rest = 0.0
        rows_layer.append({
            'layer': layer_idx,
            'emotion': emo,
            'baseline_f1': baseline_per_emotion[i],
            'compressed_f1': compressed_per_emotion[i],
            'patched_f1': layer_patched_f1[layer_idx, i],
            'restoration_score': rest,
        })

df_layer_patching = pd.DataFrame(rows_layer)
layer_csv = os.path.join(results_dir, 'activation_patching_per_layer.csv')
df_layer_patching.to_csv(layer_csv, index=False)
print(f'Saved: {layer_csv} ({len(df_layer_patching)} rows)')

# 2. Component-level patching results
rows_comp = []
for (layer_idx, component), (per_emo, f1_m) in component_patched.items():
    for i, emo in enumerate(emotion_names):
        gap = baseline_per_emotion[i] - compressed_per_emotion[i]
        safe_g = gap if abs(gap) > 0.005 else 1.0
        rest = np.clip((per_emo[i] - compressed_per_emotion[i]) / safe_g, 0, 1)
        if abs(gap) <= 0.005:
            rest = 0.0
        rows_comp.append({
            'layer': layer_idx,
            'component': component,
            'emotion': emo,
            'baseline_f1': baseline_per_emotion[i],
            'compressed_f1': compressed_per_emotion[i],
            'patched_f1': per_emo[i],
            'restoration_score': rest,
        })

df_comp_patching = pd.DataFrame(rows_comp)
comp_csv = os.path.join(results_dir, 'activation_patching_per_component.csv')
df_comp_patching.to_csv(comp_csv, index=False)
print(f'Saved: {comp_csv} ({len(df_comp_patching)} rows)')

# 3. Summary
rows_summary = [
    {'model': 'baseline', 'f1_macro': baseline_f1_macro},
    {'model': f'compressed_rank_{PATCH_RANK}', 'f1_macro': compressed_f1_macro},
]
for layer_idx in range(12):
    rows_summary.append({
        'model': f'patched_layer_{layer_idx}',
        'f1_macro': layer_patched_macro[layer_idx],
    })
for (layer_idx, component), (_, f1_m) in component_patched.items():
    rows_summary.append({
        'model': f'patched_layer_{layer_idx}_{component}',
        'f1_macro': f1_m,
    })
df_summary = pd.DataFrame(rows_summary)
summary_csv = os.path.join(results_dir, 'activation_patching_summary.csv')
df_summary.to_csv(summary_csv, index=False)
print(f'Saved: {summary_csv} ({len(df_summary)} rows)')

# ── Final report ──
print('\n' + '=' * 65)
print('ACTIVATION PATCHING STUDY COMPLETE')
print('=' * 65)
print(f'Compression rank: {PATCH_RANK}')
print(f'Baseline F1 macro:   {baseline_f1_macro:.4f}')
print(f'Compressed F1 macro: {compressed_f1_macro:.4f}')
print(f'\nLayer-level patching (F1 macro):')
for i in range(12):
    rest = (layer_patched_macro[i] - compressed_f1_macro) / max(baseline_f1_macro - compressed_f1_macro, 1e-8)
    bar = '#' * int(rest * 40)
    print(f'  Layer {i:2d}: {layer_patched_macro[i]:.4f}  '
          f'(restoration = {rest:5.1%})  {bar}')

print(f'\nMost critical layers (highest mean restoration):')
for l in top_layers:
    print(f'  Layer {l}: mean restoration = {mean_restoration_per_layer[l]:.3f}')

print(f'\nEmotions with most localized information (single critical layer):')
for vi in range(n_valid):
    rest_profile = restoration_valid[:, vi]
    if rest_profile.max() > 0.1:
        peak = np.argmax(rest_profile)
        concentration = rest_profile[peak] / max(rest_profile.sum(), 1e-8)
        if concentration > 0.4:
            print(f'  {valid_emotions[vi]:18s}: Layer {peak} '
                  f'(concentration = {concentration:.1%})')

print(f'\nFigures saved in results/:')
for fig_name in ['activation_patching_01_restoration_map.png',
                 'activation_patching_02_mean_restoration.png',
                 'activation_patching_03_component_bars.png',
                 'activation_patching_04_per_emotion_curves.png',
                 'activation_patching_05_brain_scan.png',
                 'activation_patching_06_validation.png']:
    path = os.path.join(results_dir, fig_name)
    status = 'ok' if os.path.exists(path) else 'pending'
    print(f'  [{status}] {fig_name}')
print(f'\nCSVs saved:')
print(f'  activation_patching_per_layer.csv ({len(df_layer_patching)} rows)')
print(f'  activation_patching_per_component.csv ({len(df_comp_patching)} rows)')
print(f'  activation_patching_summary.csv ({len(df_summary)} rows)')